# Project Milestone Two: Modeling and Feature Engineering

### Overview

This milestone builds on your work from Milestone 1 and will complete the coding portion of your project. You will:

1. Pick 3 modeling algorithms from those we have studied.
2. Evaluate baseline models using default settings.
3. Engineer new features and re-evaluate models.
4. Use feature selection techniques and re-evaluate.
5. Fine-tune for optimal performance.
6. Select your best model and report on your results.

You must do all work in this notebook and upload to your team leader's account in Gradescope. There is no
Individual Assessment for this Milestone.


In [13]:
# ===================================
# Useful Imports: Add more as needed
# ===================================

# Standard Libraries
import os
import time
import math
import io
import zipfile
import requests
from urllib.parse import urlparse
from itertools import chain, combinations

# Data Science Libraries
import numpy as np
import pandas as pd
import seaborn as sns

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as mticker  # Optional: Format y-axis labels as dollars
import seaborn as sns

# Scikit-learn (Machine Learning)
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV,
    RandomizedSearchCV,
    RepeatedKFold
)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.feature_selection import SequentialFeatureSelector, f_regression, SelectKBest
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, GradientBoostingRegressor

# Progress Tracking

from tqdm import tqdm

# =============================
# Global Variables
# =============================
random_state = 42

# =============================
# Utility Functions
# =============================

# Format y-axis labels as dollars with commas (optional)
def dollar_format(x, pos):
    return f'${x:,.0f}'

# Convert seconds to HH:MM:SS format
def format_hms(seconds):
    return time.strftime("%H:%M:%S", time.gmtime(seconds))



### Prelude: Load your Preprocessed Dataset from Milestone 1

In Milestone 1, you handled missing values, encoded categorical features, and explored your data. Before you begin this milestone, you’ll need to load that cleaned dataset and prepare it for modeling. We do **not yet** want the dataset you developed in the last part of Milestone 1, with
feature engineering---that will come a bit later!

Here’s what to do:

1. Return to your Milestone 1 notebook and rerun your code through Part 3, where your dataset was fully cleaned (assume it’s called `df_cleaned`).

2. **Save** the cleaned dataset to a file by running:

>   df_cleaned.to_csv("zillow_cleaned.csv", index=False)

3. Switch to this notebook and **load** the saved data:

>   df = pd.read_csv("zillow_cleaned.csv")

4. Create a **train/test split** using `train_test_split`.  
   
6. **Standardize** the features (but not the target!) using **only the training data.** This ensures consistency across models without introducing data leakage from the test set:

>   scaler = StandardScaler()   
>   X_train_scaled = scaler.fit_transform(X_train)    
  
**Notes:**

- You will have to redo the scaling step if you introduce new features (which have to be scaled as well).


In [14]:
from google.colab import drive
drive.mount("/content/drive/")

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [15]:
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/df_clean.csv")


X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=["taxvaluedollarcnt"]), df["taxvaluedollarcnt"], test_size=0.2, random_state=random_state)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

### Part 1: Picking Three Models and Establishing Baselines [6 pts]

Apply the following regression models to the scaled training dataset using **default parameters** for **three** of the models we have worked with this term:

- Linear Regression
- Ridge Regression
- Lasso Regression
- Decision Tree Regression
- Bagging
- Random Forest
- Gradient Boosting Trees

For each of the three models:
- Use **repeated cross-validation** (e.g., 5 folds, 5 repeats).
- Report the **mean and standard deviation of CV MAE Score**.


In [41]:
# Linear Regression

def run_linear_regression(X_train_scaled, y_train,
                          cv = 5,
                          scoring = 'neg_mean_absolute_error',
                          n_jobs = None
                          ):

    # Define model
    linear_model = LinearRegression()

    # Perform cross-validation
    cv = RepeatedKFold(n_splits = 5, n_repeats = 5, random_state = random_state)
    mae_scores = -cross_val_score(linear_model, X_train_scaled, y_train,
                                  cv = RepeatedKFold(n_splits = 5, n_repeats = 5, random_state = random_state),
                                  scoring = 'neg_mean_absolute_error',
                                  n_jobs = n_jobs
                                  )

    # Compute scores
    mean_cv_mae = np.mean(mae_scores)
    std_cv_mae = np.std(mae_scores)

    # Fit model on training set
    linear_model.fit(X_train_scaled, y_train)

    # Compute training MAE
    train_pred = linear_model.predict(X_train_scaled)
    train_mae = mean_absolute_error(y_train, train_pred)

    # Compute test MAE
    # test_pred = linear_model.predict(X_test)
    # test_mae = mean_absolute_error(y_test, test_pred)

    return mean_cv_mae, std_cv_mae, train_mae

# Run model
mean_cv_mae, std_cv_mae, train_mae = run_linear_regression(X_train_scaled, y_train,
    cv = 5,
    scoring ='neg_mean_absolute_error',
    n_jobs = None
    )

# Print results
print("Linear Regression:\n")
print(f"Mean CV MAE Score: {mean_cv_mae:.4f}")
print(f"STD CV MAE Score: {std_cv_mae:.4f}")
print(f"Train MAE Score: {train_mae:.4f}")
# print(f"Test MAE Score: {test_mae:.4f}\n")

Linear Regression:

Mean CV MAE Score: 171731.3491
STD CV MAE Score: 1486.9109
Train MAE Score: 171556.8549


In [44]:
# Random Forest

def run_random_forest_regressor(X_train_scaled, y_train,
                                n_estimators = 100,
                                max_depth = None,
                                min_samples_split = 2,
                                min_samples_leaf = 1,
                                max_samples = None,
                                max_leaf_nodes = None,
                                max_features = 1.0,
                                bootstrap = True,
                                random_state = random_state,
                                cv = 5,
                                ccp_alpha = 0.0,
                                n_jobs = None
                               ):

    # Define model
    random_forest_model = RandomForestRegressor(
                          n_estimators = n_estimators,
                          max_depth = max_depth,
                          max_samples = max_samples,
                          min_samples_split = min_samples_split,
                          min_samples_leaf = min_samples_leaf,
                          max_leaf_nodes=max_leaf_nodes,
                          max_features = max_features,
                          bootstrap = bootstrap,
                          random_state = random_state,
                          ccp_alpha=ccp_alpha,
                          n_jobs = n_jobs,
                          )

    # Perform cross-validation
    cv = RepeatedKFold(n_splits = 5, n_repeats = 5, random_state = random_state)
    mae_scores = -cross_val_score(random_forest_model, X_train_scaled, y_train,
                                  cv = RepeatedKFold(n_splits = 5, n_repeats = 5, random_state = random_state),
                                  scoring = 'neg_mean_absolute_error',
                                  n_jobs = n_jobs
                                  )

    # Compute scores
    mean_cv_mae = np.mean(mae_scores)
    std_cv_mae = np.std(mae_scores)

    # Fit model on training set
    random_forest_model.fit(X_train_scaled, y_train)

    # Compute training MAE
    train_pred = random_forest_model.predict(X_train_scaled)
    train_mae = mean_absolute_error(y_train, train_pred)

    # Compute test MAE
    # test_pred = random_forest_model.predict(X_test)
    # test_mae = mean_absolute_error(y_test, test_pred)

    return mean_cv_mae, std_cv_mae, train_mae

# Run model
mean_cv_mae, std_cv_mae, train_mae = run_random_forest_regressor(X_train_scaled, y_train,
    n_estimators = 100,
    max_depth = None,
    min_samples_split = 2,
    min_samples_leaf = 1,
    max_samples = None,
    max_leaf_nodes = None,
    max_features = 1.0,
    ccp_alpha = 0.0,
    )

# Print results
print("Random Forest Regression:\n")
print(f"Mean CV MAE Score: {mean_cv_mae:.4f}")
print(f"STD CV MAE Score: {std_cv_mae:.4f}")
print(f"Train MAE Score: {train_mae:.4f}")
# print(f"Test MAE Score: {test_mae:.4f}\n")

Random Forest Regression:

Mean CV MAE Score: 149854.2583
STD CV MAE Score: 1219.7500
Train MAE Score: 56452.4184


In [43]:
# Gradient Boosting Trees

parameters_default = {
    'learning_rate'           : 0.1,
    'n_estimators'            : 100,
    'max_depth'               : 3,
    'min_samples_split'       : 2,
    'min_samples_leaf'        : 1,
    'max_features'            : None,
    'max_leaf_nodes'          : None,
    'random_state'            : random_state,
    }

def run_gradient_boosting_regressor(X_train_scaled, y_train,
                                    parameters_default,
                                    n_repeats = 5
                                    ):

    # Define model
    gradient_boosting_model = GradientBoostingRegressor(
                              learning_rate = parameters_default['learning_rate'],
                              n_estimators = parameters_default['n_estimators'],
                              max_depth = parameters_default['max_depth'],
                              min_samples_split = parameters_default['min_samples_split'],
                              min_samples_leaf = parameters_default['min_samples_leaf'],
                              max_features = parameters_default['max_features'],
                              max_leaf_nodes = parameters_default['max_leaf_nodes'],
                              random_state = parameters_default['random_state']
                              # all other parameters left at their defaults
                              )

    # Perform cross-validation
    cv = RepeatedKFold(n_splits = 5, n_repeats = n_repeats, random_state = random_state)
    mae_scores = -cross_val_score(gradient_boosting_model, X_train_scaled, y_train,
                                  cv = RepeatedKFold(n_splits = 5, n_repeats = 5, random_state = random_state),
                                  scoring ='neg_mean_absolute_error',
                                  n_jobs = None
                                  )

    # Compute scores
    mean_cv_mae = np.mean(mae_scores)
    std_cv_mae = np.std(mae_scores)

    # Fit the model on the full training set
    gradient_boosting_model.fit(X_train_scaled, y_train)

    # Compute training MSE
    train_pred = gradient_boosting_model.predict(X_train_scaled)
    train_mae = mean_absolute_error(y_train, train_pred)

    # Compute test MSE
    # test_pred = gradient_boosting_model.predict(X_test)
    # test_mae = mean_squared_error(y_test, test_pred)

    return mean_cv_mae, std_cv_mae, train_mae

# Run model
mean_cv_mae, std_cv_mae, train_mae = run_gradient_boosting_regressor(X_train_scaled, y_train,
    parameters_default,
    n_repeats = 5
    )

# Print results
print("Gradient Boosting Trees Regressor:\n")
print(f"Mean CV MAE Score: {mean_cv_mae:.4f}")
print(f"STD CV MAE Score: {std_cv_mae:.4f}")
print(f"Train MAE Score: {train_mae:.4f}")
# print(f"Test MAE Score: {test_mae:.4f}\n")


Gradient Boosting Trees Regressor:

Mean CV MAE Score: 154347.2102
STD CV MAE Score: 1281.9887
Train MAE Score: 152857.8110


### Part 1: Discussion [3 pts]

In a paragraph or well-organized set of bullet points, briefly compare and discuss:

  - Which model performed best overall?
  - Which was most stable (lowest std)?
  - Any signs of overfitting or underfitting?

Three different regression models were chosen, using default parameters, in order to establish baseline cross-validated mean absolute error values. The three regression models chosen include: a linear regression model, a random forest regression model, and a gradient boosting trees regression model.

When comparing the overall performance of the three models, the random forest regression model appears to perform the best overall. That is, the random forest model's observed CV MAE (STD) score of 1219.7500 is significanlty lower than those of the gradient boosting trees regressor model's CV MAE (STD) score of 1281.9887, and the linear regression model's CV MAE (STD) score of 1486.9109. These values indicate the random forest regression model is the most stable model.

As for any signs of overfitting or underfitting, each model shows a mean CV MAE and a training MAE score that is relatively close to each other, suggesting little to no overfitting. However, given that each model was trained using default parameters, the issue of underfitting may exist as the models are not complex enough. Adjusting the hyperperameters in order to fine-tune each model may help address this potential issue.

### Part 2: Feature Engineering [6 pts]

Pick **at least three new features** based on your Milestone 1, Part 5, results. You may pick new ones or
use the same ones you chose for Milestone 1.

Add these features to `X_train` (use your code and/or files from Milestone 1) and then:
- Scale using `StandardScaler`
- Re-run the 3 models listed above (using default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  


In [ ]:


# --- Feature Engineering Functions (from Milestone 1, Part 5) ---

def add_log_lot_size(df):
    df = df.copy()
    df["log_lotsizesquarefeet"] = np.log1p(df["lotsizesquarefeet"])
    return df

def add_home_age(df, reference_year=2017):
    df = df.copy()
    df["home_age"] = reference_year - df["yearbuilt"]
    return df

def add_bath_per_bedroom(df):
    df = df.copy()
    df["bath_per_bedroom"] = df["bathroomcnt"] / df["bedroomcnt"].replace(0, np.nan)
    df["bath_per_bedroom"] = df["bath_per_bedroom"].fillna(df["bathroomcnt"])
    return df

def apply_feature_engineering(df):
    df = add_log_lot_size(df)
    df = add_home_age(df)
    df = add_bath_per_bedroom(df)
    df = df.drop(columns=["yearbuilt"])  # replaced by home_age
    return df

# Apply to train and test sets
X_train_fe = apply_feature_engineering(X_train)
X_test_fe  = apply_feature_engineering(X_test)

# Re-scale with StandardScaler fit on training data only
scaler_fe = StandardScaler()
X_train_fe_scaled = scaler_fe.fit_transform(X_train_fe)

print(f"New X_train shape: {X_train_fe.shape}")
print(f"New features:  log_lotsizesquarefeet, home_age, bath_per_bedroom")
print(f"Dropped:       yearbuilt (replaced by home_age)")


In [ ]:

# Linear Regression - with engineered features

mean_cv_mae_lr_fe, std_cv_mae_lr_fe, train_mae_lr_fe = run_linear_regression(X_train_fe_scaled, y_train)

print("Linear Regression (with engineered features):\n")
print(f"Mean CV MAE Score: {mean_cv_mae_lr_fe:.4f}")
print(f"STD CV MAE Score:  {std_cv_mae_lr_fe:.4f}")
print(f"Train MAE Score:   {train_mae_lr_fe:.4f}")


In [ ]:

# Random Forest - with engineered features

mean_cv_mae_rf_fe, std_cv_mae_rf_fe, train_mae_rf_fe = run_random_forest_regressor(X_train_fe_scaled, y_train)

print("Random Forest Regression (with engineered features):\n")
print(f"Mean CV MAE Score: {mean_cv_mae_rf_fe:.4f}")
print(f"STD CV MAE Score:  {std_cv_mae_rf_fe:.4f}")
print(f"Train MAE Score:   {train_mae_rf_fe:.4f}")


Random Forest Regression (with engineered features):

Mean CV MAE Score: 150019.3931
STD CV MAE Score:  1194.2568
Train MAE Score:   56450.6664


In [ ]:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                            
# Gradient Boosting Trees - with engineered features

mean_cv_mae_gb_fe, std_cv_mae_gb_fe, train_mae_gb_fe = run_gradient_boosting_regressor(X_train_fe_scaled, y_train, parameters_default)

print("Gradient Boosting Trees Regressor (with engineered features):\n")
print(f"Mean CV MAE Score: {mean_cv_mae_gb_fe:.4f}")
print(f"STD CV MAE Score:  {std_cv_mae_gb_fe:.4f}")
print(f"Train MAE Score:   {train_mae_gb_fe:.4f}")


### Part 2: Discussion [3 pts]

Reflect on the impact of your new features:

- Did any models show notable improvement in performance?

- Which new features seemed to help — and in which models?

- Do you have any hypotheses about why a particular feature helped (or didn’t)?




None of the models showed a notable improvement in performance after adding the new features. The only meaningful directional change was in Linear Regression, with the mean CV MAE slightly decreasing by roughly 0.16%. Both the Random Forest and Gradient Boosting moved slightly in the wrong direction. Of the three new features, 'log_lotsizesquarefeet' is the best candidate for a small improvement in Linear Regression. 'lotsizesquarefeet' is strongly right-skewed and the log transform compresses the long tail and linearizes the relationship with the target. Tree-based models split on thresholds and indifferent to distribution will receive no benefit from this kind of transformation. 'home_age' is essentially a linear rescaling of 'yearbuilt', so that it carries identical information. 'bath_per_bedroom' brings in a ratio interaction that linear regression cannot derive on its own. Additionally, tree-based models can approximate the ratio through sequential splits without needing it explicity. 

Overall, the results are consistent with the Milestone 1 F-score analysis, that showed none of the engineered features exceeded the strongest original predictors. Meaning the original feature set was already well-represented for models at default settings, which leaves little room for improvement with the transformations experimented. 

### Part 3: Feature Selection [6 pts]

Using the full set of features (original + engineered):
- Apply **feature selection** methods to investigate whether you can improve performance.
  - You may use forward selection, backward selection, or feature importance from tree-based models.
- For each model, identify the **best-performing subset of features**.
- Re-run each model using only those features (with default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  


In [ ]:
# Add as many cells as you need


### Part 3: Discussion [3 pts]

Analyze the effect of feature selection on your models:

- Did performance improve for any models after reducing the number of features?

- Which features were consistently retained across models?

- Were any of your newly engineered features selected as important?


> Your text here

### Part 4: Fine-Tuning Your Three Models [6 pts]

In this final phase of Milestone 2, you’ll select and refine your **three most promising models and their corresponding data pipelines** based on everything you've done so far, and pick a winner!

1. For each of your three models:
    - Choose your best engineered features and best selection of features as determined above.
   - Perform hyperparameter tuning using `sweep_parameters`, `GridSearchCV`, `RandomizedSearchCV`, `Optuna`, etc. as you have practiced in previous homeworks.
3. Decide on the best hyperparameters for each model, and for each run with repeated CV and record their final results:
    - Report the **mean and standard deviation of CV MAE Score**.  

In [ ]:
# Add as many cells as you need


### Part 4: Discussion [3 pts]

Reflect on your tuning process and final results:

- What was your tuning strategy for each model? Why did you choose those hyperparameters?
- Did you find that certain types of preprocessing or feature engineering worked better with specific models?


> Your text here

### Part 5: Final Model and Design Reassessment [6 pts]

In this part, you will finalize your best-performing model.  You’ll also consolidate and present the key code used to run your model on the preprocessed dataset.
**Requirements:**

- Decide one your final model among the three contestants.

- Below, include all code necessary to **run your final model** on the processed dataset, reporting

    - Mean and standard deviation of CV MAE Score.
    
    - Test score on held-out test set.




In [ ]:
# Add as many cells as you need


### Part 5: Discussion [8 pts]

In this final step, your goal is to synthesize your entire modeling process and assess how your earlier decisions influenced the outcome. Please address the following:

1. Model Selection:
- Clearly state which model you selected as your final model and why.

- What metrics or observations led you to this decision?

- Were there trade-offs (e.g., interpretability vs. performance) that influenced your choice?

2. Revisiting an Early Decision

- Identify one specific preprocessing or feature engineering decision from Milestone 1 (e.g., how you handled missing values, how you scaled or encoded a variable, or whether you created interaction or polynomial terms).

- Explain the rationale for that decision at the time: What were you hoping it would achieve?

- Now that you've seen the full modeling pipeline and final results, reflect on whether this step helped or hindered performance. Did you keep it, modify it, or remove it?

- Justify your final decision with evidence—such as validation scores, visualizations, or model diagnostics.

3. Lessons Learned

- What insights did you gain about your dataset or your modeling process through this end-to-end workflow?

- If you had more time or data, what would you explore next?

> Your text here